In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import os
os.chdir("/Users/lior/DS6021/DS6021_F26/data")

auto = pd.read_csv("Auto.csv", na_values="?").dropna().reset_index(drop=True)
num = ["displacement", "horsepower", "weight", "acceleration", "year"]
X = auto[num]; y = auto["mpg"]
SEED = 200
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
alphas = np.logspace(-3, 3, 100)
auto.shape
#used claude to clean and help with code

(392, 9)

In [3]:
table = pd.DataFrame(index=num)
ols_all = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
table["OLS_all"] = ols_all[-1].coef_
sub = ["horsepower","weight","year"]
ols_3 = make_pipeline(StandardScaler(),LinearRegression()).fit(auto[sub], y)
table["OLS_3"] = pd.Series(ols_3[-1].coef_, index=sub)
table.round(4)

,OLS_all,OLS_3
displacement,0.2907,NaN
horsepower,0.0392,-0.1922
weight,-5.8312,-5.4697
acceleration,0.2489,NaN
year,2.7744,2.7545


1b

ridge and lasso adds penalty to the size of the coefficients and the size of the coeffs depends on the variable units so without standardizing the penalty would shrink values unequally because of the units. standardizing puts the predictors on the same scale so the penalty treats everything equally and ols has no penalty so rescaling a variable just rescales the coefficient and the fitted values are similar

In [4]:
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=cv)).fit(X, y)
print("Ridge alpha_:", ridge[-1].alpha_)
table["Ridge"] = ridge[-1].coef_
table.round(4)

Ridge alpha_: 1.232846739442066


,OLS_all,OLS_3,Ridge
displacement,0.2907,NaN,0.1473
horsepower,0.0392,-0.1922,-0.0444
weight,-5.8312,-5.4697,-5.6340
acceleration,0.2489,NaN,0.2011
year,2.7744,2.7545,2.7527


2b

ridge chose alpha = 1.233

2c

ridge shrinks but never goes to zero. wright changes most absolte terms and displacement seems to be basically split in hald and horsepower flips the sign, and those are the correlated variables whose ols estimates were unstable

In [5]:
lasso = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=cv)).fit(X, y)
print("Lasso alpha_:", lasso[-1].alpha_)
table["Lasso"] = lasso[-1].coef_
table.round(4)

Lasso alpha_: 0.049770235643321115


,OLS_all,OLS_3,Ridge,Lasso
displacement,0.2907,NaN,0.1473,-0.0000
horsepower,0.0392,-0.1922,-0.0444,-0.0000
weight,-5.8312,-5.4697,-5.6340,-5.5324
acceleration,0.2489,NaN,0.2011,0.1542
year,2.7744,2.7545,2.7527,2.7207


3b

lasso went with alpha = 0.4977 and set displacement and horsepower to 0 and the rest are kept

3c

lasso considers all features all at once in one fit and it chooses the amount of shrinkage by cross validation to optimize out of sample error so that the selection. is more stable and not greedy. the limitations is with correlated predictors lasso tends to pick one randomly so the selected set can change in a samole and the ones that are kept are biased towards zero. it also only captures linear effects and doesnt giv standard errors or pvalues.

In [6]:
enet = make_pipeline(StandardScaler(),
                     ElasticNetCV(alphas=alphas, l1_ratio=[0.1, 0.5, 0.9, 1], cv=cv)).fit(X, y)
print("alpha_:", enet[-1].alpha_, " l1_ratio_:", enet[-1].l1_ratio_)
table["ElasticNet"] = enet[-1].coef_
table.round(4)

alpha_: 0.049770235643321115  l1_ratio_: 1.0


,OLS_all,OLS_3,Ridge,Lasso,ElasticNet
displacement,0.2907,NaN,0.1473,-0.0000,-0.0000
horsepower,0.0392,-0.1922,-0.0444,-0.0000,-0.0000
weight,-5.8312,-5.4697,-5.6340,-5.5324,-5.5324
acceleration,0.2489,NaN,0.2011,0.1542,0.1542
year,2.7744,2.7545,2.7527,2.7207,2.7207


4b

alpha = 0.04977 and l1 ratio = 1.0. cross validation prefers a lasso penality so elasticnets coefficients are identical to lassos

4c

weignt and year are the important feature because their coefficients dont really chance across OLS, ridge, lasso, etc..acceleration is small but positive. displacement and horsepower disagree with theirmethods because lasso and elasticnet drop them and ride leaves them small which means their info is redundant when correlated with the weight